# Import `deepseek_part1.json` vao Neo4j bang Cypher

Notebook nay doc file JSON, chuan hoa payload, sau do dung `UNWIND` + `MERGE` de import Part -> Chapter -> Crime -> Rule -> Condition -> Penalty vao Neo4j.

In [2]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

candidate_env_paths = [
    Path(".env"),
    Path("chatbot_rag/.env"),
]
for env_path in candidate_env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        break
else:
    load_dotenv(override=True)

JSON_PATH = Path("chatbot_rag/deepseek_part1.json")
if not JSON_PATH.exists():
    JSON_PATH = Path("deepseek_part1.json")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

if not NEO4J_URI or not NEO4J_USER or not NEO4J_PASSWORD:
    raise ValueError("Thieu bien moi truong NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD trong .env")

if not JSON_PATH.exists():
    raise FileNotFoundError(f"Khong tim thay file JSON tai: {JSON_PATH}")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connected to Neo4j at {NEO4J_URI}")
print(f"JSON path: {JSON_PATH.resolve()}")

Connected to Neo4j at neo4j+s://cd9c5c25.databases.neo4j.io
JSON path: C:\Users\Admin\Desktop\DATN\chatbot_rag\deepseek_part1.json


In [3]:
with JSON_PATH.open("r", encoding="utf-8") as f:
    raw_data = json.load(f)

if "parts" in raw_data:
    raw_parts = raw_data["parts"]
elif "part" in raw_data and "chapters" in raw_data:
    raw_parts = [raw_data]
else:
    raise KeyError("JSON phai co key 'parts' hoac dong thoi co 'part' va 'chapters'.")

import_rows = []
for raw_part in raw_parts:
    part_meta = raw_part["part"]
    for raw_chapter in raw_part.get("chapters", []):
        for raw_article in raw_chapter.get("articles", []):
            rules_payload = []
            for raw_rule in raw_article.get("rules", []):
                conditions_payload = []
                for idx, raw_condition in enumerate(raw_rule.get("conditions", []), start=1):
                    value = raw_condition.get("value")
                    conditions_payload.append({
                        "id": f"{raw_rule['rule_id']}_cond_{idx}",
                        "type": raw_condition.get("type"),
                        "field": raw_condition.get("field"),
                        "operator": raw_condition.get("operator"),
                        "min": value.get("min") if isinstance(value, dict) else None,
                        "max": value.get("max") if isinstance(value, dict) else None,
                        "value": value if not isinstance(value, dict) else None,
                        "unit": raw_condition.get("unit"),
                        "text": raw_condition.get("text")
                    })

                raw_penalty = raw_rule.get("penalty", {}) or {}
                penalty_payload = {
                    "id": f"{raw_rule['rule_id']}_penalty",
                    "min": raw_penalty.get("min"),
                    "max": raw_penalty.get("max"),
                    "fine_min": raw_penalty.get("fine_min"),
                    "fine_max": raw_penalty.get("fine_max"),
                    "extra": raw_penalty.get("extra"),
                    "note": raw_penalty.get("note")
                }

                rules_payload.append({
                    "id": raw_rule["rule_id"],
                    "clause": raw_rule.get("clause"),
                    "logic": raw_rule.get("logic"),
                    "priority": raw_rule.get("priority"),
                    "conditions": conditions_payload,
                    "penalty": penalty_payload
                })

            crime = raw_article["crime"]
            import_rows.append({
                "part": {
                    "id": str(part_meta["part_id"]),
                    "name": part_meta.get("name")
                },
                "chapter": {
                    "id": str(raw_chapter["chapter_id"]),
                    "name": raw_chapter.get("name")
                },
                "crime": {
                    "id": str(crime["id"]),
                    "name": crime.get("name"),
                    "article": crime.get("article")
                },
                "rules": rules_payload
            })

print(f"So article rows se import: {len(import_rows)}")
if import_rows:
    print(f"Part: {import_rows[0]['part']['name']}")

So article rows se import: 107
Part: Những quy định chung


In [4]:
CONSTRAINT_QUERIES = [
    "CREATE CONSTRAINT part_id IF NOT EXISTS FOR (n:Part) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT chapter_id IF NOT EXISTS FOR (n:Chapter) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT crime_id IF NOT EXISTS FOR (n:Crime) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT rule_id IF NOT EXISTS FOR (n:Rule) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT condition_id IF NOT EXISTS FOR (n:Condition) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT penalty_id IF NOT EXISTS FOR (n:Penalty) REQUIRE n.id IS UNIQUE"
]

IMPORT_QUERY = """
UNWIND $rows AS row
MERGE (p:Part {id: row.part.id})
SET p.name = row.part.name
MERGE (c:Chapter {id: row.chapter.id})
SET c.name = row.chapter.name
MERGE (p)-[:HAS_CHAPTER]->(c)
MERGE (cr:Crime {id: row.crime.id})
SET cr.name = row.crime.name,
    cr.article = row.crime.article
MERGE (c)-[:HAS_ARTICLE]->(cr)
WITH cr, row
UNWIND row.rules AS rule
MERGE (r:Rule {id: rule.id})
SET r.clause = rule.clause,
    r.logic = rule.logic,
    r.priority = rule.priority
MERGE (cr)-[:HAS_RULE]->(r)
WITH r, rule
FOREACH (cond IN rule.conditions |
    MERGE (cnd:Condition {id: cond.id})
    SET cnd.type = cond.type,
        cnd.field = cond.field,
        cnd.operator = cond.operator,
        cnd.min = cond.min,
        cnd.max = cond.max,
        cnd.value = cond.value,
        cnd.unit = cond.unit,
        cnd.text = cond.text
    MERGE (r)-[:HAS_CONDITION]->(cnd)
)
WITH r, rule
MERGE (pn:Penalty {id: rule.penalty.id})
SET pn.min = rule.penalty.min,
    pn.max = rule.penalty.max,
    pn.fine_min = rule.penalty.fine_min,
    pn.fine_max = rule.penalty.fine_max,
    pn.extra = rule.penalty.extra,
    pn.note = rule.penalty.note
MERGE (r)-[:HAS_PENALTY]->(pn)
"""

def run_write(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        summary = result.consume()
        return summary.counters

def run_write_in_batches(query, rows, batch_size=10):
    totals = {
        "nodes_created": 0,
        "relationships_created": 0,
        "properties_set": 0,
    }
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        counters = run_write(query, {"rows": batch})
        totals["nodes_created"] += counters.nodes_created
        totals["relationships_created"] += counters.relationships_created
        totals["properties_set"] += counters.properties_set
        print(f"Imported batch {start // batch_size + 1}: {len(batch)} articles")
    return totals

for query in CONSTRAINT_QUERIES:
    run_write(query)

print("Constraints ready")

Constraints ready


In [5]:
counters = run_write_in_batches(IMPORT_QUERY, import_rows, batch_size=10)
print(counters)

Imported batch 1: 10 articles
Imported batch 2: 10 articles
Imported batch 3: 10 articles
Imported batch 4: 10 articles
Imported batch 5: 10 articles
Imported batch 6: 10 articles
Imported batch 7: 10 articles
Imported batch 8: 10 articles
Imported batch 9: 10 articles
Imported batch 10: 10 articles
Imported batch 11: 7 articles
{'nodes_created': 1013, 'relationships_created': 1012, 'properties_set': 4142}


In [6]:
CHECK_QUERY = """
MATCH (p:Part)
OPTIONAL MATCH (p)-[:HAS_CHAPTER]->(c:Chapter)
OPTIONAL MATCH (c)-[:HAS_ARTICLE]->(cr:Crime)
OPTIONAL MATCH (cr)-[:HAS_RULE]->(r:Rule)
OPTIONAL MATCH (r)-[:HAS_CONDITION]->(cd:Condition)
OPTIONAL MATCH (r)-[:HAS_PENALTY]->(pn:Penalty)
RETURN count(DISTINCT p) AS parts,
       count(DISTINCT c) AS chapters,
       count(DISTINCT cr) AS crimes,
       count(DISTINCT r) AS rules,
       count(DISTINCT cd) AS conditions,
       count(DISTINCT pn) AS penalties
"""

with driver.session() as session:
    summary = session.run(CHECK_QUERY).single()
    print(dict(summary))

{'parts': 1, 'chapters': 12, 'crimes': 107, 'rules': 244, 'conditions': 405, 'penalties': 244}


In [7]:
# Dong ket noi sau khi import xong
driver.close()